In [3]:
import pandas as pd
import sqlite3
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.nn.utils.rnn import pack_padded_sequence
torch.manual_seed(112358)

In [ ]:
conn=sqlite3.connect("./archive.sqlite3")
cursor=conn.cursor()
df=pd.read_sql_query("SELECT * FROM posts,post_tags where posts.thread_id=post_tags.thread_id",conn)

DatabaseError: Execution failed on sql 'SELECT * FROM posts,post_tags where posts.thread_id=post_tags.thread_id': database disk image is malformed

In [ ]:
df

In [ ]:
df=df[df.is_first_post==1]
df

In [ ]:
df.memory_usage(deep=True)

In [ ]:
tag=df["tag"].value_counts()
tag

In [ ]:
# we 'll take the first 4
tag=tag[tag>10000]
tag

In [ ]:
df=df[df.tag.isin(tag.index)]

In [ ]:
df

In [ ]:
ds=df.loc[:, ~df.columns.duplicated()].groupby("thread_id",as_index=True).agg(
    text=("processed_html","first"),
    tag=("tag",list)
)
ds

In [4]:
import ast
ds=pd.read_csv("out.csv")
ds["tag"]=ds["tag"].apply(ast.literal_eval)
ds

,thread_id,text,tag
0,2,"Let $ABC$ be a triangle, and $M$ an interior p...",[geometry]
1,3,okay this one is from Prof. Mircea Lascu from ...,"[algebra, geometry]"
2,5,"If A,B are invertible and the set {Ak - Bk | k...",[algebra]
3,9,In a magic square $n \times n$ composed from t...,"[algebra, combinatorics]"
4,72,The lengths of the sides of a convex hexagon $...,[geometry]
...,...,...,...
46451,36238654,"Let $a, b, c$ be the altitudes of triangle $A$...",[geometry]
46452,36238689,Find all functions that satisfy the condition ...,[algebra]
46453,36238706,On an $N \times N$ “chessboard” ($N \ge 3$) ea...,[combinatorics]
46454,36238733,It is known that $(20 + 25)^2 = 2025$. Find al...,[number theory]


In [5]:
ds.memory_usage(deep=True)

,0
Index,132
thread_id,371648
text,21120721
tag,4585376


In [6]:
sp = spm.SentencePieceProcessor(model_file="my_tokenizer.model")

In [7]:
def tokenize(text):
    return sp.encode(text,out_type=int)
X=ds["text"].apply(tokenize)
X

,text
0,"[215, 3, 427, 7916, 81, 6, 332, 7929, 35, 3, 7..."
1,"[4808, 232, 149, 275, 29, 264, 387, 7924, 7937..."
2,"[320, 79, 7929, 7951, 101, 7886, 35, 9, 439, 4..."
3,"[622, 6, 7885, 471, 3, 7914, 8, 705, 45, 7916,..."
4,"[266, 2315, 31, 9, 927, 31, 6, 2166, 3058, 3, ..."
...,...
46451,"[215, 3, 7912, 7929, 24, 7929, 18, 7916, 81, 9..."
46452,"[1022, 170, 1892, 38, 1023, 9, 733, 98, 170, 5..."
46453,"[1782, 22, 3, 7967, 8, 705, 147, 7916, 7909, 0..."
46454,"[533, 29, 1689, 38, 156, 1611, 68, 2514, 256, ..."


In [8]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(ds["tag"])
print(mlb.classes_)

['algebra' 'combinatorics' 'geometry' 'number theory']


In [9]:
class ContestProblemDataset(Dataset):
    def __init__(self, X, Y):
        self.X = [torch.tensor(x, dtype=torch.long) for x in X]
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

def collate_fn(batch):
    xs, ys = zip(*batch)
    x_lens = torch.tensor([len(x) for x in xs])
    x_padded = pad_sequence(list(xs), batch_first=True, padding_value=8000)
    y_batch = torch.stack(ys)
    return x_padded, y_batch, x_lens

In [10]:
BATCH_SIZE=32

X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=42)
X_val,X_test,Y_val,Y_test=train_test_split(X_test,Y_test,test_size=0.5,random_state=42)

train_ds=ContestProblemDataset(X_train,Y_train)
val_ds=ContestProblemDataset(X_val,Y_val)
test_ds=ContestProblemDataset(X_test,Y_test)

train_loader=DataLoader(train_ds,BATCH_SIZE,shuffle=True,collate_fn=collate_fn)
val_loader=DataLoader(val_ds,BATCH_SIZE,shuffle=False,collate_fn=collate_fn)
test_loader=DataLoader(test_ds,BATCH_SIZE,shuffle=False,collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 1017
Validation batches: 218
Test batches: 218


In [11]:
# train_ds.X

In [12]:
class CategoryLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout_rate, padding_idx):
        super().__init__()

        # 1. Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        # UPGRADE 1: Embedding Dropout (Prevents overfitting on specific words)
        self.embed_dropout = nn.Dropout(dropout_rate)

        # UPGRADE 2: Fix LSTM Dropout logic
        # PyTorch ignores LSTM dropout if n_layers == 1.
        lstm_dropout = dropout_rate if n_layers > 1 else 0.0

        self.LSTM = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            dropout=lstm_dropout,
            batch_first=True
        )

        # UPGRADE 3: Two-layer Classifier Head (Much better accuracy)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(2 * hidden_dim, hidden_dim), # Step down gently
            nn.ReLU(),                             # Activation function
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, output_dim)      # Final raw logits output
        )

    def forward(self, text_batch, x_lens):
        # Apply embedding and dropout
        embedded = self.embed_dropout(self.embedding(text_batch))

        # Pack the sequence (Excellent choice for performance!)
        packed = pack_padded_sequence(embedded, x_lens.cpu(), batch_first=True, enforce_sorted=False)

        # LSTM forward pass
        packed_out, (hidden, cell) = self.LSTM(packed)

        # hidden shape: [num_layers * num_directions, batch, hidden_dim]
        # Get the last hidden state of both directions
        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]

        # Concatenate forward and backward
        bi_hidden = torch.cat((hidden_forward, hidden_backward), dim=1)

        # Pass through upgraded classifier head
        linear_out = self.classifier(bi_hidden)

        return linear_out
VOCAB_SIZE = 8001
EMBEDDING_DIM = 128  # 128 is faster on GPUs than 100
HIDDEN_DIM = 128     # BiLSTM doubles this to 256. (256 hidden -> 512 total is too large for 46k samples)
OUTPUT_DIM = 4       # Number of classes
N_LAYERS = 1         # Keep as 1 for fast training on 100-200 tokens
DROPOUT = 0.4        # 0.5 is sometimes a bit too harsh for a 1-layer LSTM; 0.3-0.4 is the sweet spot
PADDING_IDX = 8000

# Instantiate the model
model = CategoryLSTM(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    n_layers=N_LAYERS,
    dropout_rate=DROPOUT,
    padding_idx=PADDING_IDX
)

print(model)

CategoryLSTM(
  (embedding): Embedding(8001, 128, padding_idx=8000)
  (embed_dropout): Dropout(p=0.4, inplace=False)
  (LSTM): LSTM(128, 128, batch_first=True, bidirectional=True)
  (classifier): Sequential(
    (0): Dropout(p=0.4, inplace=False)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=4, bias=True)
  )
)


In [13]:
# --- Training Hyperparameters ---
LEARNING_RATE = 0.001
N_EPOCHS = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=LEARNING_RATE)

model=model.to(device)
criteron=criterion.to(device)


def jaccard_accuracy(y_true, logits, threshold=0.5):
    probs = torch.sigmoid(logits)
    y_pred_bin = (probs > threshold).float()

    intersection = (y_true * y_pred_bin).sum(dim=1)
    union = ((y_true + y_pred_bin) > 0).float().sum(dim=1)
    jaccard = torch.where(union > 0, intersection / union, torch.ones_like(union))
    return jaccard.mean()

Training on device: cuda


In [14]:


for epoch in range(N_EPOCHS):

    train_loss = 0.0
    train_acc = 0.0

    # --- Training Phase ---
    model.train() # Set model to training mode

    for inputs, labels,x_len in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} [Train]"):
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Zero gradients
        optimizer.zero_grad()

        # 2. Forward pass
        predictions = model(inputs,x_len)

        # 3. Calculate loss and accuracy
        loss = criterion(predictions, labels)
        acc = jaccard_accuracy(labels,predictions)

        # 4. Backward pass
        loss.backward()

        # 5. Update weights
        optimizer.step()

        train_loss += loss.item()
        train_acc += acc.item()

    # --- Validation Phase ---
    val_loss = 0.0
    val_acc = 0.0

    model.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculation
        for inputs, labels,x_len in tqdm(val_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} [Val]"):
            inputs, labels = inputs.to(device), labels.to(device)

            predictions = model(inputs,x_len)

            loss = criterion(predictions, labels)
            acc = jaccard_accuracy(labels, predictions)

            val_loss += loss.item()
            val_acc += acc.item()

    # Print epoch statistics
    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = train_acc / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = val_acc / len(val_loader)

    print(f'Epoch {epoch+1:02} | Train Loss: {avg_train_loss:.3f} | Train Acc: {avg_train_acc*100:.2f}% | Val. Loss: {avg_val_loss:.3f} | Val. Acc: {avg_val_acc*100:.2f}%')

print("Training finished.")

Epoch 1/10 [Val]: 100%|██████████| 218/218 [00:02<00:00, 73.79it/s]


Epoch 01 | Train Loss: 0.338 | Train Acc: 63.21% | Val. Loss: 0.243 | Val. Acc: 79.88%


Epoch 2/10 [Val]: 100%|██████████| 218/218 [00:06<00:00, 36.32it/s]


Epoch 02 | Train Loss: 0.243 | Train Acc: 80.01% | Val. Loss: 0.255 | Val. Acc: 80.76%


Epoch 3/10 [Val]: 100%|██████████| 218/218 [00:03<00:00, 61.59it/s]


...

Epoch 08 | Train Loss: 0.163 | Train Acc: 87.33% | Val. Loss: 0.206 | Val. Acc: 85.50%


Epoch 9/10 [Val]: 100%|██████████| 218/218 [00:02<00:00, 75.02it/s]


Epoch 09 | Train Loss: 0.157 | Train Acc: 87.70% | Val. Loss: 0.213 | Val. Acc: 86.02%


Epoch 10/10 [Val]: 100%|██████████| 218/218 [00:02<00:00, 75.03it/s]

Epoch 10 | Train Loss: 0.152 | Train Acc: 88.33% | Val. Loss: 0.203 | Val. Acc: 85.89%
Training finished.


In [15]:
# # torch.save(model.state_dict(),"weight.model")
# model.load_state_dict(torch.load("weight.model"))

# model


In [19]:
import numpy as np
from sklearn.metrics import jaccard_score, f1_score, precision_score, recall_score
def evaluate_model(model, test_loader, criterion, device):
    model.eval()

    test_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels, x_len in tqdm(test_loader, desc="[Test]"):
            inputs, labels = inputs.to(device), labels.to(device)

            predictions = model(inputs, x_len)
            loss = criterion(predictions, labels)
            test_loss += loss.item()

            probs = torch.sigmoid(predictions)
            preds = (probs >= 0.5).int()

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_test_loss = test_loss / len(test_loader)

    Y_pred = np.concatenate(all_preds, axis=0)
    Y_test = np.concatenate(all_labels, axis=0).astype(int)

    label_acc = np.mean(Y_pred == Y_test, axis=0)

    jaccard_samples = jaccard_score(Y_test, Y_pred, average="samples", zero_division=0)
    f1_micro = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    f1_macro = f1_score(Y_test, Y_pred, average="macro", zero_division=0)
    precision_micro = precision_score(Y_test, Y_pred, average="micro", zero_division=0)
    recall_micro = recall_score(Y_test, Y_pred, average="micro", zero_division=0)

    results = {
        "test_loss": avg_test_loss,
        **{f"acc_label_{i}": float(acc) for i, acc in enumerate(label_acc)},
        "jaccard_samples": float(jaccard_samples),
        "f1_micro": float(f1_micro),
        "f1_macro": float(f1_macro),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
    }

    print(f"Test Loss: {avg_test_loss:.3f} | Jaccard: {jaccard_samples*100:.2f}% | F1 micro: {f1_micro*100:.2f}%")

    return results, Y_pred, Y_test


results, Y_pred, Y_test = evaluate_model(model, test_loader, criterion, device)
print(results)

[Test]: 100%|██████████| 218/218 [00:10<00:00, 20.37it/s]

Test Loss: 0.196 | Jaccard: 86.05% | F1 micro: 87.22%
{'test_loss': 0.19610735726192458, 'acc_label_0': 0.9223704979193571, 'acc_label_1': 0.925814320562491, 'acc_label_2': 0.9563782465203042, 'acc_label_3': 0.9147653895824365, 'jaccard_samples': 0.860513225235567, 'f1_micro': 0.8722403657740039, 'f1_macro': 0.8582338552185403, 'precision_micro': 0.8882532925369163, 'recall_micro': 0.8567945592198126}


In [16]:
model.eval()
text="""Let $\\mathbb{N}$ denote the set of positive integers. A function $f\\colon\\mathbb{N}\\to\\mathbb{N}$ is said to be bonza if
\\[
f(a)~~\\text{divides}~~b^a-f(b)^{f(a)}
\\]for all positive integers $a$ and $b$.

Determine the smallest real constant $c$ such that $f(n)\\leqslant cn$ for all bonza functions $f$ and all positive integers $n$."""
tokens = sp.encode(text, out_type=int)
inp = torch.tensor(tokens).unsqueeze(0).to(device)
x_lens = torch.tensor([len(tokens)])
print(x_lens)
with torch.no_grad():
    prediction = model(inp, x_lens)
    probs = torch.sigmoid(prediction)
    print(probs)
    predicted_labels = (probs > 0.5).int()
    predicted_indices = torch.nonzero(predicted_labels.squeeze()).flatten().tolist()
    predicted_names = [mlb.classes_[i] for i in predicted_indices]
    print(predicted_names)  # ví dụ: ['algebra', 'geometry']

tensor([110])
tensor([[0.6965, 0.0030, 0.0041, 0.5180]], device='cuda:0')
['algebra', 'number theory']


In [17]:
torch.cuda.reset_peak_memory_stats()

model = model.cuda()

print(
    f"VRAM allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)


VRAM allocated: 0.04 GB


In [21]:
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "val_loss": avg_val_loss,
}, "lstm_checkpoint.pt")